In [1]:

# Parte 2 - VulnerabilidadesIAdata
# Comprobación inicial de Python

import sys

print("Proyecto: VulnerabilidadesIAdata")
print("Versión de Python:", sys.version.split()[0])
print("Entorno preparado")


Proyecto: VulnerabilidadesIAdata
Versión de Python: 3.14.2
Entorno preparado



# VulnerabilidadesIAdata
## Parte 2 — Análisis y predicción de vulnerabilidades CVE

**Dataset:** NVD CVE-2025 (NIST)

## 1. Resumen ejecutivo

En esta segunda parte de mi proyecto voy a trabajar
con datos reales de vulnerabilidades informáticas
para intentar predecir qué nivel de gravedad
pueden tener.

La idea es ayudar a las empresas a saber qué
vulnerabilidades deberían revisar primero,
ya que no todas tienen la misma importancia
ni tienen el mismo riesgo.

Para ello, voy a analizar los datos, crear
gráficos y probar diferentes modelos de
Machine Learning para ver cuál funciona mejor.

También voy a comparar los resultados para
comprobar si estos modelos pueden ayudar
a tomar decisiones en ciberseguridad.

Y al final, explicaré los resultados que he
obtenido, los posibles errores del modelo
y cómo se podría utilizar en una empresa.



## 2. Del problema al Machine Learning

En la primera parte expliqué cómo
la inteligencia artificial podría ayudar a las
empresas a gestionar mejor las vulnerabilidades.

Ahora quiero comprobar si es posible
predecir la gravedad de una vulnerabilidad
utilizando la información que tenemos sobre ella.

Voy a utilizar el dataset NVD CVE-2025,
que contiene información sobre vulnerabilidades
informáticas.

Mi objetivo es crear un modelo
que pueda distinguir entre vulnerabilidades
de gravedad baja, media, alta y crítica.

Esto podría ayudar a los responsables de
ciberseguridad a organizar mejor su trabajo
y decidir qué vulnerabilidades revisar primero.

La predicción será una ayuda para tomar decisiones,
pero no sustituirá la revisión de una mano humana.



### 1.1. Definición del modelo

| Pregunta | Mi proyecto |
|---|---|
| ¿Qué decisión quiero hacer mejor? | Ayudar a decidir qué vulnerabilidades revisar primero. |
| ¿Qué quiero predecir? | La gravedad de una vulnerabilidad: baja, media, alta o crítica. |
| ¿Qué representa cada registro? | Una vulnerabilidad identificada mediante un código CVE. |
| ¿Cuándo se realizará la predicción? | Cuando se registre una vulnerabilidad y todavía no tenga una clasificación definitiva. |
| ¿Qué información utilizaré? | La descripción de la vulnerabilidad y otras características disponibles en el momento. |
| ¿Cómo mediré los resultados? | Con métricas como precision, recall, F1-score... |
| ¿Qué beneficio espero conseguir? | Ayudar a los equipos de seguridad a identificar antes las vulnerabilidades más importantes. |



## 3. Datos y calidad del dataset

Ahora voy a analizar los datos que
utilizaré para entrenar mi modelo de Machine Learning.

Mi dataset viene de de la National Vulnerability
Database (NVD) y contiene información sobre
vulnerabilidades informáticas registradas con
identificadores CVE.

El archivo original contiene 45.373 registros.

Antes de entrenar el modelo, voy a comprobar
qué información tiene cada vulnerabilidad,
si hay datos que faltan y si existen registros
dobles osea duplicados.

También revisaré cuáles tienen una gravedad
asignada y cuáles no.

Este análisis me ayudará a preparar los datos
y evitar problemas que puedan afectar
a las predicciones.


In [4]:

# Cargar el dataset original de vulnerabilidades CVE

import json
from pathlib import Path

ruta = Path("data/nvdcve-2.0-2025.json")

with open(ruta, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

print("Dataset cargado correctamente")
print("Total de vulnerabilidades:", len(datos["vulnerabilities"]))
print("Formato:", datos["format"])
print("Versión:", datos["version"])


Dataset cargado correctamente
Total de vulnerabilidades: 45373
Formato: NVD_CVE
Versión: 2.0


In [5]:

# Comprobar duplicados y clasificaciones

import pandas as pd

registros = []

for elemento in datos["vulnerabilities"]:
    cve = elemento["cve"]

    registros.append({
        "cve_id": cve.get("id"),
        "fecha": cve.get("published"),
        "estado": cve.get("vulnStatus"),
        "tiene_cvss": any(
            clave.startswith("cvssMetric")
            for clave in cve.get("metrics", {})
        )
    })

df = pd.DataFrame(registros)

print("Total de registros:", len(df))
print("CVE duplicados:", df["cve_id"].duplicated().sum())
print("Registros con métricas CVSS:", df["tiene_cvss"].sum())
print("Registros sin métricas CVSS:", (~df["tiene_cvss"]).sum())

display(df.head())


Total de registros: 45373
CVE duplicados: 0
Registros con métricas CVSS: 43176
Registros sin métricas CVSS: 2197


,cve_id,fecha,estado,tiene_cvss
0,CVE-2025-0168,2025-01-01T14:15:23.590,Analyzed,True
1,CVE-2025-22214,2025-01-02T04:15:06.277,Deferred,True
2,CVE-2025-0171,2025-01-02T15:15:25.550,Analyzed,True
3,CVE-2025-0172,2025-01-02T16:15:09.103,Analyzed,True
4,CVE-2025-0173,2025-01-02T18:15:21.630,Analyzed,True


In [3]:

# Comprobar las librerías que se necesitan para el proyecto

import json
import pandas as pd
import numpy as np
import plotly.express as px
import sklearn
import joblib

print("Todas las librerías se han cargado correctamente")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)


Todas las librerías se han cargado correctamente
Pandas: 3.1.1
NumPy: 2.5.5
Scikit-learn: 1.9.1



### 3.1 Primeros resultados del análisis

Después de cargar el dataset,
contiene 45.373 vulnerabilidades.

De todas ellas, 43.176 tienen alguna métrica
CVSS y 2.197 no tienen esta información.

Esto significa que mas o menos el 95,16 %
de los registros cuentan con datos de evaluación
CVSS, mientras que al 4,84 % les falta.

También he comprobado que no hay identificadores
CVE duplicados, por lo tanto es positivo porque
evita contar dos veces la misma vulnerabilidad.

Esto me ayuda a conocer mejor
la calidad de los datos antes de entrenar
los modelos.

### ¿Qué significa esto para una empresa?

Para una empresa pues es necesario el saber qué
vulnerabilidades tienen información suficiente
para poder analizarlas.

Las que no tienen una evaluación CVSS necesitarán
una revisión adicional, ya que no podemos
dar por hecho su nivel de gravedad.


In [6]:

# Gráfico de disponibilidad de métricas CVSS

resumen_cvss = (
    df["tiene_cvss"]
    .map({True: "Con CVSS", False: "Sin CVSS"})
    .value_counts()
    .rename_axis("Estado")
    .reset_index(name="Cantidad")
)

fig = px.bar(
    resumen_cvss,
    x="Estado",
    y="Cantidad",
    color="Estado",
    text="Cantidad",
    title="Disponibilidad de métricas CVSS en el dataset",
    labels={
        "Estado": "Disponibilidad de CVSS",
        "Cantidad": "Número de vulnerabilidades"
    }
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()


In [7]:

# Ver la gravedad de las vulnerabilidades

def obtener_gravedad(cve):
    metricas = cve.get("metrics", {})

    # Priorizar la evaluación principal de NVD,
    # cuando esté disponible.
    for version in [
        "cvssMetricV31",
        "cvssMetricV30",
        "cvssMetricV40"
    ]:
        evaluaciones = metricas.get(version, [])

        if evaluaciones:
            principal = next(
                (
                    m for m in evaluaciones
                    if m.get("type") == "Primary"
                    and m.get("source") == "nvd@nist.gov"
                ),
                evaluaciones[0]
            )

            return principal.get(
                "cvssData", {}
            ).get("baseSeverity")

    return None


# Extraer la gravedad de todos los registros
df["gravedad"] = [
    obtener_gravedad(item["cve"])
    for item in datos["vulnerabilities"]
]

print("Distribución de vulnerabilidades:")
print(df["gravedad"].fillna("SIN CLASIFICAR").value_counts())


Distribución de vulnerabilidades:
gravedad
MEDIUM            19678
HIGH              15988
CRITICAL           6304
SIN CLASIFICAR     2197
LOW                1190
NONE                 16
Name: count, dtype: int64


In [8]:

# Gráfico de vulnerabilidades por gravedad

orden = [
    "LOW", "MEDIUM", "HIGH",
    "CRITICAL", "NONE", "SIN CLASIFICAR"
]

conteo = (
    df["gravedad"]
    .fillna("SIN CLASIFICAR")
    .value_counts()
    .reindex(orden, fill_value=0)
    .reset_index()
)

conteo.columns = ["Gravedad", "Cantidad"]

fig = px.bar(
    conteo,
    x="Gravedad",
    y="Cantidad",
    color="Gravedad",
    text="Cantidad",
    title="Distribución de vulnerabilidades por gravedad",
    labels={
        "Gravedad": "Nivel de gravedad",
        "Cantidad": "Número de vulnerabilidades"
    }
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()



### 3.2 Análisis de la gravedad

Después de analizar los datos, he visto que
la mayoría de las vulnerabilidades tienen
una gravedad media o alta.

En concreto, hay 19.678 vulnerabilidades
de gravedad media, 15.988 altas y 6.304 críticas.

También hay 1.190 de gravedad baja,
2.197 sin clasificar y 16 con gravedad NONE.

Me he fijado bastante en que hay muchas
vulnerabilidades altas y críticas, por lo que
es importante que las empresas tengan una forma
de organizar cuáles deben revisar primero.

Estos resultados muestran que una empresa puede
tener muchas vulnerabilidades que revisar.

Así es que este modelo predictivo podría ayudar a detectar
antes las que parecen más graves, pero siempre
sería necesario comprobar el contexto de cada
empresa antes de decidir cuáles solucionar primero.


In [9]:

# Comprobar datos que falten

# Extraer las descripciones en inglés
descripciones = []

for elemento in datos["vulnerabilities"]:
    cve = elemento["cve"]

    descripcion = next(
        (
            d.get("value")
            for d in cve.get("descriptions", [])
            if d.get("lang") == "en"
        ),
        None
    )

    descripciones.append(descripcion)

df["descripcion"] = descripciones

# Comprobar los valores que no estan
columnas = [
    "cve_id",
    "fecha",
    "estado",
    "descripcion",
    "gravedad"
]

faltantes = df[columnas].isna().sum()

print("Datos que faltan por columna:")
print(faltantes)

print("\nTotal de registros:", len(df))


Datos que faltan por columna:
cve_id            0
fecha             0
estado            0
descripcion       0
gravedad       2197
dtype: int64

Total de registros: 45373



### 3.3 Comprobación de datos que no estan

He revisado las columnas del dataset
para ver si faltaba información.

Los resultados muestran que los identificadores CVE,
las fechas, los estados y las descripciones
no tienen valores vacíos.

Sin embargo, hay 2.197 vulnerabilidades que
no tienen una gravedad asignada.

También he encontrado 16 registros con gravedad NONE,
que no forman parte de las cuatro categorías
que quiero predecir.

Por eso, para entrenar los modelos utilizaré
los registros que tengan una gravedad válida:
LOW, MEDIUM, HIGH o CRITICAL.

No voy a inventar cualquier gravedad para los registros
que no la tienen, porque eso podría afectar
a los resultados del modelo.

In [10]:

# Seleccionar las cuatro categorías de gravedad

categorias_validas = [
    "LOW",
    "MEDIUM",
    "HIGH",
    "CRITICAL"
]

# Crear una copia con registros válidos
df_limpio = df[
    df["gravedad"].isin(categorias_validas)
].copy()

# Convertir las fechas a formato fecha
df_limpio["fecha"] = pd.to_datetime(
    df_limpio["fecha"],
    errors="coerce"
)

print("Registros originales:", len(df))
print("Registros preparados:", len(df_limpio))
print("Registros excluidos:", len(df) - len(df_limpio))

print("\nDistribución final:")
print(df_limpio["gravedad"].value_counts())

print("\nFechas no válidas:")
print(df_limpio["fecha"].isna().sum())


Registros originales: 45373
Registros preparados: 43160
Registros excluidos: 2213

Distribución final:
gravedad
MEDIUM      19678
HIGH        15988
CRITICAL     6304
LOW          1190
Name: count, dtype: int64

Fechas no válidas:
0



### 3.4 Limpieza de los datos

Después de revisar los datos, he preparado una
versión del dataset para utilizarla en los modelos.

De los 45.373 registros originales, he conservado 43.160 porque tienen una gravedad válida.

He quitado 2.213 registros: 2.197 que no tenían
una gravedad asignada y 16 que aparecían como NONE.

No he eliminado estos registros del archivo original, solo los he como separado de los datos que voy a utilizar para entrenar el modelo.

También he convertido las fechas al formato adecuado y he comprobado que no hay fechas inválidas.

Con esta limpieza puedo trabajar con datos que
tienen una clasificación conocida y evaluar mejor las predicciones del modelo.

Aun así, las vulnerabilidades sin clasificación
siguen siendo importantes y deberían revisarse
igualmente por separado.


In [11]:

# Analizar las fechas del dataset

df_limpio["anio"] = df_limpio["fecha"].dt.year
df_limpio["mes"] = df_limpio["fecha"].dt.to_period("M").astype(str)

print("Primera fecha:", df_limpio["fecha"].min())
print("Última fecha:", df_limpio["fecha"].max())

print("\nVulnerabilidades por año:")
print(df_limpio["anio"].value_counts().sort_index())

print("\nVulnerabilidades por mes:")
print(df_limpio["mes"].value_counts().sort_index())


Primera fecha: 2025-01-01 14:15:23.590000
Última fecha: 2026-10-15 09:17:14.190000

Vulnerabilidades por año:
anio
2025    37977
2026     5183
Name: count, dtype: int64

Vulnerabilidades por mes:
mes
2025-01    1854
2025-02    1793
2025-03    2921
2025-04    3691
2025-05    3226
2025-06    3209
2025-07    3582
2025-08    3349
2025-09    3730
2025-10    3617
2025-11    2855
2025-12    4150
2026-01    1991
2026-02    1081
2026-03     640
2026-04     271
2026-05     337
2026-06     392
2026-07     161
2026-08     150
2026-09     112
2026-10      48
Name: count, dtype: int64


In [14]:
# Evolución mes a mes de las vulnerabilidades

vulnerabilidades_mes = (
    df_limpio.groupby("mes")
    .size()
    .reset_index(name="cantidad")
    .sort_values("mes")
)

fig = px.line(
    vulnerabilidades_mes,
    x="mes",
    y="cantidad",
    markers=True,
    title="Evolución mensual de vulnerabilidades CVE",
    labels={
        "mes": "Mes de publicación",
        "cantidad": "Número de vulnerabilidades"
    }
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()



### 3.5 Análisis de las fechas

He revisado las fechas de publicación de las
vulnerabilidades para comprobar cómo se
distribuyen a lo largo del tiempo, el dataset contiene registros publicados tanto en 2025 como en 2026, aunque el nombre del archivo haga referencia a 2025.

Con el gráfico puedo ver cómo cambia la cantidad de vulnerabilidades publicadas cada mes.

Esto es importante porque, cuando entrene
los modelos, utilizaré los datos más antiguos
para entrenar y los más recientes para comprobar si las predicciones funcionan.

Las vulnerabilidades nuevas aparecen continuamente, por lo que una empresa necesita cosas que puedan analizar información reciente.

Por eso quiero comprobar que mi modelo funciona
también con vulnerabilidades pasadas a las
que utilizó para aprender.



## 4. Qué he descubierto al analizar los datos

Me interesa comprobar cuáles son los tipos
de debilidades más frecuentes y si pueden
tener alguna relación con la gravedad.

Esto también me ayudará a decidir qué información podría utilizar para entrenar los modelos de Machine Learning.


In [17]:

# Extraer los códigos CWE de las vulnerabilidades

def obtener_cwe(cve):
    codigos = []

    for debilidad in cve.get("weaknesses", []):
        for descripcion in debilidad.get("description", []):
            valor = descripcion.get("value", "")

            if valor.startswith("CWE-") and valor not in codigos:
                codigos.append(valor)

    return codigos[0] if codigos else "SIN CWE"


df["cwe"] = [
    obtener_cwe(elemento["cve"])
    for elemento in datos["vulnerabilities"]
]

df_limpio["cwe"] = df.loc[df_limpio.index, "cwe"]

print("Los 10 códigos CWE más frecuentes:")
print(df_limpio["cwe"].value_counts().head(10))


Los 10 códigos CWE más frecuentes:
cwe
CWE-79     7611
CWE-74     2566
CWE-862    2234
CWE-352    1720
CWE-89     1567
SIN CWE    1351
CWE-119    1153
CWE-22     1040
CWE-284     993
CWE-125     831
Name: count, dtype: int64


In [18]:

# Gráfico de los 10 CWE más frecuentes

top_cwe = (
    df_limpio["cwe"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_cwe.columns = ["CWE", "Cantidad"]

fig = px.bar(
    top_cwe,
    x="CWE",
    y="Cantidad",
    color="CWE",
    text="Cantidad",
    title="Los 10 tipos de debilidades más frecuentes",
    labels={
        "CWE": "Código de debilidad",
        "Cantidad": "Número de vulnerabilidades"
    }
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()



### 4.1 Tipos de debilidades más frecuentes

Después de analizar los códigos CWE, he visto que el más frecuente es CWE-79, relacionado con
vulnerabilidades Cross-Site Scripting (XSS),
con 7.611 registros.

También aparecen otros tipos de problemas,
como las inyecciones, los fallos de autorización y los errores de control de acceso.

Esto me ha ayudado a entender que no todas las
vulnerabilidades son iguales y que pueden
afectar a diferentes partes de un sistema.

Para este análisis he utilizado el primer código CWE disponible en cada registro, aunque algunas vulnerabilidades pueden tener varios.

In [19]:

# Relación entre los tipos CWE y la gravedad

principales_cwe = (
    df_limpio["cwe"]
    .value_counts()
    .head(10)
    .index
)

datos_cwe = df_limpio[
    df_limpio["cwe"].isin(principales_cwe)
]

tabla_cwe_gravedad = pd.crosstab(
    datos_cwe["cwe"],
    datos_cwe["gravedad"],
    normalize="index"
) * 100

tabla_cwe_gravedad = tabla_cwe_gravedad.reindex(
    columns=["LOW", "MEDIUM", "HIGH", "CRITICAL"],
    fill_value=0
)

display(tabla_cwe_gravedad.round(2))


gravedad,LOW,MEDIUM,HIGH,CRITICAL
cwe,,,,
CWE-119,3.21,12.06,60.02,24.72
CWE-125,4.33,40.55,51.14,3.97
CWE-22,2.50,37.79,43.17,16.54
CWE-284,3.02,37.66,39.48,19.84
CWE-352,0.70,56.45,41.22,1.63
CWE-74,0.23,5.38,30.67,63.72
CWE-79,2.15,76.93,20.01,0.91
CWE-862,1.30,76.19,18.62,3.89
CWE-89,0.51,20.55,46.84,32.10


In [20]:

# Gráfico de gravedad por tipo de CWE

tabla_grafico = (
    tabla_cwe_gravedad
    .reset_index()
    .melt(
        id_vars="cwe",
        var_name="Gravedad",
        value_name="Porcentaje"
    )
)

fig = px.bar(
    tabla_grafico,
    x="cwe",
    y="Porcentaje",
    color="Gravedad",
    barmode="stack",
    title="Distribución de gravedad según el tipo de CWE",
    labels={
        "cwe": "Tipo de debilidad CWE",
        "Porcentaje": "Porcentaje de vulnerabilidades (%)"
    },
    category_orders={
        "Gravedad": ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_range=[0, 100]
)

fig.show()



### 4.2 Relación entre las debilidades y la gravedad

He comparado los tipos de debilidades CWE con
los niveles de gravedad de las vulnerabilidades.

Los resultados muestran que existen diferencias
bastante grandes según el tipo de problema.

Por ejemplo, el 63,72 % de las vulnerabilidades
CWE-74 son críticas, mientras que en CWE-79
la mayoría son de gravedad media (76,93 %).

También he visto que el 32,10 % de las
vulnerabilidades CWE-89 son críticas.

Esto me hace pensar que el tipo de debilidad
podría ser una variable útil para mi modelo
de Machine Learning.

Sin embargo, todavía tengo que comprobarlo
entrenando los modelos y comparando sus resultados.

Aun así, no se debe decidir la prioridad
de una vulnerabilidad solo por su CWE,
porque también influyen otros factores,
como los sistemas afectados y otros motivos.



### 4.3 Comprobación de las hipótesis

En la primera parte de mi proyecto planteé
varias hipótesis sobre las vulnerabilidades
y el uso de Machine Learning.

Ahora voy a comprobarlas utilizando los
datos reales del dataset.

**Hipótesis 1: Relación entre el vector de ataque y la gravedad**

Quiero comprobar si las vulnerabilidades que
se pueden explotar a través de la red suelen
tener una gravedad mayor que las que necesitan
otro tipo de acceso.

Para ello, voy a comparar los niveles de gravedad según el vector de ataque de cada vulnerabilidad.


In [21]:

# Hipótesis 1: vector de ataque y gravedad

def obtener_vector_ataque(cve):
    metricas = cve.get("metrics", {})

    for version in [
        "cvssMetricV31",
        "cvssMetricV30",
        "cvssMetricV40"
    ]:
        evaluaciones = metricas.get(version, [])

        if evaluaciones:
            principal = next(
                (
                    m for m in evaluaciones
                    if m.get("type") == "Primary"
                    and m.get("source") == "nvd@nist.gov"
                ),
                evaluaciones[0]
            )

            return principal.get(
                "cvssData", {}
            ).get("attackVector")

    return None


# Añadir el vector de ataque
df["vector_ataque"] = [
    obtener_vector_ataque(item["cve"])
    for item in datos["vulnerabilities"]
]

df_limpio["vector_ataque"] = df.loc[
    df_limpio.index, "vector_ataque"
]

# Calcular porcentaje de vulnerabilidades altas y críticas
tabla_vector = pd.crosstab(
    df_limpio["vector_ataque"],
    df_limpio["gravedad"],
    normalize="index"
) * 100

tabla_vector = tabla_vector.reindex(
    columns=["LOW", "MEDIUM", "HIGH", "CRITICAL"],
    fill_value=0
)

tabla_vector["HIGH + CRITICAL (%)"] = (
    tabla_vector["HIGH"] + tabla_vector["CRITICAL"]
)

display(tabla_vector.round(2))


gravedad,LOW,MEDIUM,HIGH,CRITICAL,HIGH + CRITICAL (%)
vector_ataque,,,,,
ADJACENT,3.90,34.42,43.51,18.18,61.69
ADJACENT_NETWORK,8.84,48.88,40.27,2.01,42.28
LOCAL,4.47,47.69,47.39,0.46,47.85
NETWORK,2.03,44.63,34.70,18.64,53.34
PHYSICAL,13.37,77.80,8.83,0.00,8.83


In [22]:

# Unificar los nombres de vectores

df_limpio["vector_ataque_unificado"] = (
    df_limpio["vector_ataque"]
    .replace({
        "ADJACENT_NETWORK": "ADJACENT"
    })
)

# Calcular cuántas vulnerabilidades hay en cada grupo
resumen_vector = (
    df_limpio
    .dropna(subset=["vector_ataque_unificado"])
    .groupby("vector_ataque_unificado")
    .agg(
        total=("gravedad", "size"),
        altas_criticas=(
            "gravedad",
            lambda x: x.isin(["HIGH", "CRITICAL"]).sum()
        )
    )
)

resumen_vector["porcentaje"] = (
    resumen_vector["altas_criticas"]
    / resumen_vector["total"] * 100
)

display(resumen_vector.round(2))


,total,altas_criticas,porcentaje
vector_ataque_unificado,,,
ADJACENT,1048,473,45.13
LOCAL,8331,3986,47.85
NETWORK,33362,17796,53.34
PHYSICAL,419,37,8.83



### 4.4 Resultado de la hipótesis 1

Después de comparar los diferentes vectores
de ataque, he visto que las vulnerabilidades
que se pueden explotar por la red tienen
un porcentaje más alto de gravedad alta o crítica.

En concreto, el 53,34 % de las vulnerabilidades
de tipo NETWORK son altas o críticas.

En comparación, las de tipo LOCAL tienen
un 47,85 %, las ADJACENT un 45,13 % y
las PHYSICAL un 8,83 %.

Aun así, esto no significa que todas las
vulnerabilidades de red sean más graves,
sino que en este dataset aparecen con
una mayor proporción de gravedad alta o crítica.


### 4.5 Resumen de las hipótesis

| Hipótesis | Estado | Evidencia |
|---|---|---|
| H1. Vector de ataque y gravedad | Respaldada | NETWORK tiene un 53,34 % de vulnerabilidades altas o críticas, por encima de los demás vectores. |
| H2. Tipo de debilidad y gravedad | Respaldada | CWE-74 tiene un 63,72 % de vulnerabilidades críticas, frente al 0,91 % de CWE-79. |
| H3. Clasificación mediante IA | Pendiente | Todavía tengo que entrenar los modelos y comparar sus resultados. |
| H4. Indicadores de rendimiento | Parcialmente evaluada | Ya he calculado la cantidad de vulnerabilidades y la cobertura CVSS, pero faltan las métricas de los modelos. |



## 5. Variables que voy a utilizar en el modelo

En este apartado voy a seleccionar la información que utilizará mi modelo de Machine Learning para intentar predecir la gravedad de las vulnerabilidades.

Para empezar, utilizaré principalmente la
descripción de cada vulnerabilidad, ya que
contiene información sobre el fallo de seguridad.

También estudiaré si el tipo de debilidad
(CWE) puede mejorar las predicciones.

Sin embargo, tengo que comprobar si esa información estaba disponible en el momento de la predicción.

La idea es que una empresa pueda obtener una
estimación inicial de gravedad a partir de
la información disponible sobre una
vulnerabilidad nueva.

Esta predicción serviría como apoyo mientras
se realiza una evaluación de seguridad más
completa.


In [23]:

# Analizar las descripciones de las vulnerabilidades

df_limpio["longitud_descripcion"] = (
    df_limpio["descripcion"].str.len()
)

print("Longitud media de las descripciones:")
print(round(df_limpio["longitud_descripcion"].mean(), 2))

print("\nDescripción más corta:")
print(df_limpio["longitud_descripcion"].min())

print("\nDescripción más larga:")
print(df_limpio["longitud_descripcion"].max())

print("\nPrimeros ejemplos:")
display(
    df_limpio[
        ["cve_id", "descripcion", "gravedad"]
    ].head(5)
)


Longitud media de las descripciones:
388.14

Descripción más corta:
15

Descripción más larga:
3998

Primeros ejemplos:


,cve_id,descripcion,gravedad
0,CVE-2025-0168,A vulnerability classified as critical has bee...,HIGH
1,CVE-2025-22214,Landray EIS 2001 through 2006 allows Message/f...,MEDIUM
2,CVE-2025-0171,"A vulnerability, which was classified as criti...",HIGH
3,CVE-2025-0172,A vulnerability has been found in code-project...,HIGH
4,CVE-2025-0173,A vulnerability was found in SourceCodester On...,HIGH


In [24]:

# Buscar palabras que podrían revelar la gravedad

patron = (
    r"\b(critical|high severity|medium severity|"
    r"low severity|cvss score|cvss base score|"
    r"cvss:3\.[01]|cvss:4\.0)\b"
)

df_limpio["texto_con_gravedad"] = (
    df_limpio["descripcion"]
    .str.contains(patron, case=False, regex=True, na=False)
)

total = df_limpio["texto_con_gravedad"].sum()

print("Descripciones con posibles pistas de gravedad:", total)
print("Porcentaje:", round(total / len(df_limpio) * 100, 2), "%")


/tmp/ipykernel_8776/227860728.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(patron, case=False, regex=True, na=False)


Descripciones con posibles pistas de gravedad: 3282
Porcentaje: 7.6 %



### 5.1 Revisión de las descripciones

He revisado las descripciones de las vulnerabilidades para comprobar si contienen palabras que puedan dar pistas sobre su gravedad.

He encontrado 3.282 descripciones con expresiones relacionadas con niveles de gravedad o CVSS, lo que representa aproximadamente el 7,60 % de los registros preparados.

Esto es importante porque no quiero que mi modelo aprenda simplemente a reconocer palabras como
"critical" o puntuaciones CVSS.

Mi objetivo es que aprenda a partir de la
información técnica de las vulnerabilidades.


In [25]:

# Preparar descripciones
import re

patron_cvss = (
    r"\b(?:critical|high severity|medium severity|"
    r"low severity|cvss score|cvss base score)\b"
    r"|CVSS:[234]\.[0-9](?:/[A-Za-z0-9_:.-]+)*"
)

def limpiar_descripcion(texto):
    if not isinstance(texto, str):
        return ""

    texto = re.sub(
        patron_cvss,
        " ",
        texto,
        flags=re.IGNORECASE
    )

    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


df_limpio["descripcion_limpia"] = (
    df_limpio["descripcion"].apply(limpiar_descripcion)
)

print("Descripciones preparadas:", len(df_limpio))
print("Ejemplo original:")
print(df_limpio["descripcion"].iloc[0][:250])

print("\nEjemplo preparado:")
print(df_limpio["descripcion_limpia"].iloc[0][:250])


Descripciones preparadas: 43160
Ejemplo original:
A vulnerability classified as critical has been found in code-projects Job Recruitment 1.0. This affects an unknown part of the file /_parse/_feedback_system.php. The manipulation of the argument person leads to sql injection. It is possible to initi

Ejemplo preparado:
A vulnerability classified as has been found in code-projects Job Recruitment 1.0. This affects an unknown part of the file /_parse/_feedback_system.php. The manipulation of the argument person leads to sql injection. It is possible to initiate the a


In [26]:

# Comprobar el resultado de la limpieza

modificadas = (
    df_limpio["descripcion"]
    != df_limpio["descripcion_limpia"]
).sum()

vacias = (
    df_limpio["descripcion_limpia"]
    .str.strip()
    .eq("")
    .sum()
)

print("Total de descripciones:", len(df_limpio))
print("Descripciones modificadas:", modificadas)
print("Descripciones vacías:", vacias)

print("\nEjemplos de descripciones:")
display(
    df_limpio[
        ["cve_id", "descripcion_limpia"]
    ].head(5)
)


Total de descripciones: 43160
Descripciones modificadas: 10603
Descripciones vacías: 0

Ejemplos de descripciones:


,cve_id,descripcion_limpia
0,CVE-2025-0168,A vulnerability classified as has been found i...
1,CVE-2025-22214,Landray EIS 2001 through 2006 allows Message/f...
2,CVE-2025-0171,"A vulnerability, which was classified as , was..."
3,CVE-2025-0172,A vulnerability has been found in code-project...
4,CVE-2025-0173,A vulnerability was found in SourceCodester On...



### 5.2 Resultado de la limpieza del texto

Después de preparar las descripciones, he comprobado que sigo teniendo 43.160 registros.

También he comprobado que ninguna descripción
ha quedado vacía después de la limpieza.

He realizado esta preparación para intentar
evitar que el modelo utilice referencias
directas a la gravedad CVSS.

Aun así, esta primera limpieza no garantiza
que se hayan eliminado todas las posibles
pistas sobre la clasificación.



### 4.3. Variables seleccionadas

| Variable | Cómo la voy a utilizar | Por qué puede ayudar |
|---|---|---|
| Descripción limpia | Convertiré el texto en números utilizando TF-IDF. | Puede contener información sobre el fallo de seguridad. |
| CWE | Lo estudiaré como una variable adicional, si se confirma que estaba disponible a tiempo. | Permite identificar el tipo de debilidad. |
| Fecha de publicación | La utilizaré para ordenar y dividir el dataset. | Evita entrenar con información posterior a los datos de prueba. |
| Gravedad CVSS | Será la variable objetivo. | Es la categoría que quiero predecir. |

No voy a utilizar directamente la puntuación
CVSS ni sus componentes para entrenar,
porque podrían revelar la respuesta.

Primero voy a probar modelos basados en
la descripción de las vulnerabilidades.



## 6. División de los datos y modelo de referencia

Antes de entrenar los modelos, voy a dividir
los datos en tres grupos diferentes.

Utilizaré el 70 % para entrenamiento,
el 15 % para validación y el 15 % para prueba.

Como mi dataset tiene fechas de publicación,
voy a ordenar las vulnerabilidades desde
las más antiguas hasta las más recientes.

De esta forma, el modelo aprenderá con
vulnerabilidades pasadas y después
comprobaré cómo funciona con otras posteriores.

También crearé un modelo de referencia
sencillo para comparar sus resultados con
los modelos de Machine Learning.

In [27]:

# Dividir los datos respetando el orden temporal

df_ordenado = (
    df_limpio
    .sort_values(["fecha", "cve_id"])
    .reset_index(drop=True)
)

total = len(df_ordenado)

fin_train = int(total * 0.70)
fin_val = int(total * 0.85)

train = df_ordenado.iloc[:fin_train].copy()
val = df_ordenado.iloc[fin_train:fin_val].copy()
test = df_ordenado.iloc[fin_val:].copy()

print("Entrenamiento:", len(train))
print("Validación:", len(val))
print("Prueba:", len(test))
print("Total:", len(train) + len(val) + len(test))

print("\nFechas de cada grupo:")
for nombre, grupo in [
    ("Entrenamiento", train),
    ("Validación", val),
    ("Prueba", test)
]:
    print(
        nombre,
        "| Desde:", grupo["fecha"].min(),
        "| Hasta:", grupo["fecha"].max()
    )

print("\nDistribución de gravedad por grupo (%):")
distribucion = pd.DataFrame({
    "Entrenamiento": train["gravedad"].value_counts(normalize=True) * 100,
    "Validación": val["gravedad"].value_counts(normalize=True) * 100,
    "Prueba": test["gravedad"].value_counts(normalize=True) * 100
}).fillna(0)

display(distribucion.round(2))


Entrenamiento: 30211
Validación: 6475
Prueba: 6474
Total: 43160

Fechas de cada grupo:
Entrenamiento | Desde: 2025-01-01 14:15:23.590000 | Hasta: 2025-10-27 02:15:49.457000
Validación | Desde: 2025-10-27 02:15:49.607000 | Hasta: 2025-12-18 08:16:09.343000
Prueba | Desde: 2025-12-18 08:16:09.473000 | Hasta: 2026-10-15 09:17:14.190000

Distribución de gravedad por grupo (%):


,Entrenamiento,Validación,Prueba
gravedad,,,
MEDIUM,45.45,48.05,43.79
HIGH,36.50,36.79,39.84
CRITICAL,15.32,12.53,13.38
LOW,2.73,2.64,3.00


In [28]:

# Comprobar que la división temporal es correcta

print("¿Entrenamiento termina antes que validación?")
print(train["fecha"].max() < val["fecha"].min())

print("\n¿Validación termina antes que prueba?")
print(val["fecha"].max() < test["fecha"].min())

print("\nCVE repetidos entre conjuntos:")
print(
    len(
        set(train["cve_id"])
        & set(val["cve_id"])
    )
    +
    len(
        set(train["cve_id"])
        & set(test["cve_id"])
    )
    +
    len(
        set(val["cve_id"])
        & set(test["cve_id"])
    )
)


¿Entrenamiento termina antes que validación?
True

¿Validación termina antes que prueba?
True

CVE repetidos entre conjuntos:
0
